In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import StepLR

from itertools import product
from sklearn.model_selection import KFold

from sklearn.metrics import cohen_kappa_score
import csv
import os
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertModel, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
from sklearn.metrics import cohen_kappa_score
import numpy as np
from transformers import BertTokenizer






In [5]:
import pandas as pd
import io
from google.colab import files

# Upload the file manually


# Read the uploaded file using io.BytesIO
  # Ensure this matches your file's name


df = pd.read_excel(r'/content/drive/My Drive/dataset.xlsx')

# Display the DataFrame
display(pd.DataFrame(df))



,essay_id,prompt_id,essay_text,holistic,content,organization,word_choice,sentence_fluency,conventions,prompt_adherence,...,conjunction,pronoun,preposition,nominalization,begin_w_pronoun,begin_w_interrogative,begin_w_article,begin_w_subordination,begin_w_conjunction,begin_w_preposition
0,1,1,"""Dear local newspaper, I think effects compute...",8,4,3.0,3.0,3.0,3.0,NaN,...,0.318182,0.391304,0.470085,0.073171,0.080000,0.285714,0.000000,0.333333,0.000000,0.000000
1,2,1,"""Dear @CAPS1 @CAPS2, I believe that using comp...",9,4,4.0,4.0,3.0,4.0,NaN,...,0.409091,0.452174,0.487179,0.219512,0.160000,0.142857,0.181818,0.111111,0.000000,0.000000
2,3,1,"""Dear, @CAPS1 @CAPS2 @CAPS3 More and more peop...",7,3,3.0,3.0,4.0,4.0,NaN,...,0.363636,0.200000,0.299145,0.024390,0.040000,0.142857,0.090909,0.111111,0.100000,0.083333
3,4,1,"""Dear Local Newspaper, @CAPS1 I have found tha...",10,5,4.0,5.0,4.0,4.0,NaN,...,0.386364,0.313043,0.487179,0.390244,0.040000,0.285714,0.363636,0.222222,0.000000,0.250000
4,5,1,"""Dear @LOCATION1, I know having computers has ...",8,4,3.0,4.0,4.0,4.0,NaN,...,0.340909,0.313043,0.487179,0.268293,0.280000,0.000000,0.636364,0.444444,0.000000,0.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12971,21626,8,""" In most stories mothers and daughters are ei...",35,8,7.0,8.0,8.0,6.0,NaN,...,0.771429,0.646512,0.719424,0.133333,0.166667,0.200000,0.000000,0.363636,0.352941,0.470588
12972,21628,8,""" I never understood the meaning laughter is t...",32,7,6.0,7.0,7.0,6.0,NaN,...,0.371429,0.409302,0.446043,0.088889,0.229167,0.000000,0.111111,0.272727,0.058824,0.176471
12973,21629,8,"""When you laugh, is @CAPS5 out of habit, or is...",40,8,8.0,8.0,8.0,8.0,NaN,...,0.514286,0.427907,0.848921,0.222222,0.250000,0.400000,0.111111,0.181818,0.235294,0.235294
12974,21630,8,""" Trippin' on fe...",40,8,8.0,8.0,8.0,8.0,NaN,...,0.314286,0.451163,0.532374,0.400000,0.354167,0.200000,0.055556,0.272727,0.117647,0.294118


In [ ]:
pip install transformers datasets torch scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 17.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
pip install transformers torch scikit-learn


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from transformers import BertTokenizer, BertModel, AdamW
from transformers import get_scheduler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import cohen_kappa_score
from tqdm import tqdm

# Handle NaN predictions
def handle_nan_predictions(predictions):
    predictions = np.array(predictions)
    if np.isnan(predictions).any():
        mean_value = np.nanmean(predictions)
        predictions = np.nan_to_num(predictions, nan=mean_value)
    return predictions

# Quadratic Weighted Kappa calculation
def quadratic_weighted_kappa(y_true, y_pred):
    y_pred = handle_nan_predictions(y_pred)
    return cohen_kappa_score(y_true, np.round(y_pred), weights='quadratic')

# Custom Dataset
class EssayDataset(Dataset):
    def __init__(self, essays, features, scores, tokenizer, max_length=512):
        self.essays = essays
        self.features = features
        self.scores = scores
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.essays)

    def __getitem__(self, idx):
        essay = self.essays[idx]
        feature = self.features[idx]
        score = self.scores[idx]
        encoding = self.tokenizer(
            essay,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "features": torch.tensor(feature, dtype=torch.float),
            "score": torch.tensor(score, dtype=torch.float)
        }

# BERT Regressor Model
class BERTRegressor(nn.Module):
    def __init__(self, hidden_units=0):
        super(BERTRegressor, self).__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        feature_dim = 86  # Number of additional features
        if hidden_units > 0:
            self.fc1 = nn.Linear(self.bert.config.hidden_size + feature_dim, hidden_units)
            self.fc2 = nn.Linear(hidden_units, 1)
        else:
            self.fc1 = nn.Linear(self.bert.config.hidden_size + feature_dim, 1)
        self.relu = nn.ReLU()

    def forward(self, input_ids, attention_mask, features):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_token = outputs.pooler_output
        combined = torch.cat((cls_token, features), dim=1)
        if hasattr(self, "fc2"):
            x = self.relu(self.fc1(combined))
            x = self.fc2(x)
        else:
            x = self.fc1(combined)
        return x

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
BATCH_SIZE = 8
LEARNING_RATE = 0.005
EPOCHS = 5
L2_REG = 0.1
N_SPLITS = 7

# Load tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Assuming `df` is your DataFrame
if 'narrativity' not in df.columns:
    raise KeyError("'narrativity' column not found in the DataFrame")

# Find the index of the 'narrativity' column
narrativity_index = df.columns.get_loc('narrativity')

# Safely extract the next 86 columns after 'narrativity'
num_cols = df.shape[1]
if narrativity_index + 86 > num_cols:
    raise IndexError("Not enough columns after 'narrativity' to extract 86 columns")

# Extract features, essays, and scores
train_features = df.iloc[:, narrativity_index + 1: narrativity_index + 1 + 86].values
train_essays = df["essay_text"].tolist()
train_scores = df["holistic"].tolist()

# K-Fold Cross Validation
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
results = []

for train_index, val_index in kf.split(train_essays):
    fold_train_essays = [train_essays[i] for i in train_index]
    fold_train_features = train_features[train_index]
    fold_train_scores = [train_scores[i] for i in train_index]
    fold_val_essays = [train_essays[i] for i in val_index]
    fold_val_features = train_features[val_index]
    fold_val_scores = [train_scores[i] for i in val_index]

    train_dataset = EssayDataset(fold_train_essays, fold_train_features, fold_train_scores, tokenizer)
    val_dataset = EssayDataset(fold_val_essays, fold_val_features, fold_val_scores, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    model = BERTRegressor(hidden_units=8)
    model = model.to(device)

    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=L2_REG)
    scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=len(train_loader) * EPOCHS)

    criterion = nn.MSELoss()

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for batch in tqdm(train_loader):
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            features = batch["features"].to(device)
            targets = batch["score"].to(device)

            outputs = model(input_ids, attention_mask, features)
            loss = criterion(outputs.squeeze(), targets)
            loss.backward()
            optimizer.step()
            scheduler.step()

            train_loss += loss.item()

        model.eval()
        val_loss = 0
        predictions = []
        actuals = []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                features = batch["features"].to(device)
                targets = batch["score"].to(device)

                outputs = model(input_ids, attention_mask, features)
                val_loss += criterion(outputs.squeeze(), targets).item()
                predictions.extend(outputs.squeeze().cpu().numpy())
                actuals.extend(targets.cpu().numpy())

        qwk_score = quadratic_weighted_kappa(actuals, predictions)
        results.append(qwk_score)
        print(f"Epoch {epoch + 1}, Training Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, QWK: {qwk_score:.4f}")

print("Cross-validation results:", results)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
 35%|███▌      | 492/1391 [06:08<11:12,  1.34it/s]


KeyboardInterrupt: 